# Visualización de las últimas 20 posiciones del Ground Truth
Muestra tablero + BSPs activos para las últimas 20 posiciones del dataset de 200 juegos.

In [1]:
import numpy as np
import sys
from pathlib import Path

project_root = Path('../..').resolve()
sys.path.insert(0, str(project_root))

# Cargar datos
boards_data = np.load(project_root /'sae' / 'metrics' / '02_data' / 'board_states_1games.npz')
boards_all  = boards_data['boards']   
colors_all  = boards_data['colors']  

bsp_gt    = np.load(project_root / 'sae' / 'metrics' / '02_data' / 'bsp_ground_truth_1games.npy')       # (11800, 198)
bsp_names = np.load(project_root / 'sae' / 'metrics' / '02_data' / 'bsp_ground_truth_1games.names.npy', allow_pickle=True)

print(f'boards_all : {boards_all.shape}')
print(f'bsp_gt     : {bsp_gt.shape}')
print(f'n_bsps     : {len(bsp_names)}')

boards_all : (1, 59, 8, 8)
bsp_gt     : (59, 326)
n_bsps     : 326


In [2]:
# Últimas 15 posiciones (último juego, últimos 15 movimientos)
last15_bsp    = bsp_gt[-15:]       # (15, 198)
last15_boards = boards_all.reshape(-1, 8, 8)[-15:]   
last15_colors = colors_all.reshape(-1)[-15:]        

print(f'last15_bsp    : {last15_bsp.shape}')
print(f'last15_boards : {last15_boards.shape}')
print(f'last15_colors : {last15_colors}')

last15_bsp    : (15, 326)
last15_boards : (15, 8, 8)
last15_colors : [ 1 -1  1 -1  1 -1  1 -1  1 -1  1 -1  1 -1  1]


In [4]:
def print_board(board, color, pos_idx):
    color_str = 'Negro (1)' if color == 1 else 'Blanco (-1)'
    print(f'\n=== Posición {pos_idx+1} | Turno: {color_str} ===')
    print('  1 2 3 4 5 6 7 8')
    for i, row in enumerate(board):
        symbols = []
        for val in row:
            if val == 1:   symbols.append('●')
            elif val == -1: symbols.append('○')
            else:           symbols.append('·')
        print(f'{"abcdefgh"[i]} {" ".join(symbols)}')
    black = int((board == 1).sum())
    white = int((board == -1).sum())
    print(f'  Negras: {black}  Blancas: {white}  Total: {black+white}')


def print_active_bsps(bsp_row, bsp_names):
    active = [bsp_names[i] for i, v in enumerate(bsp_row) if v == 1]
    print(f'  BSPs activos ({len(active)}): {" ".join(active) if active else "ninguno"}')


# Mostrar las 15 posiciones
for i in range(15):
    print_board(last15_boards[i], last15_colors[i], i)
    print_active_bsps(last15_bsp[i], bsp_names)


=== Posición 1 | Turno: Negro (1) ===
  1 2 3 4 5 6 7 8
a · ● · ● · · · ·
b · ● ● ● ● ● ○ ·
c · ● ○ ○ ● ● ○ ·
d · ● ○ ● ○ ● · ○
e ● ○ ● ○ ○ ● ○ ○
f ○ ● ○ ● ○ ○ ● ○
g · ○ ● ○ ○ ● ○ ○
h ○ ● · ○ ● ● · ○
  Negras: 24  Blancas: 25  Total: 49
  BSPs activos (114): BSPA10 BSPA21 BSPA30 BSPA41 BSPA50 BSPA60 BSPA70 BSPA80 BSPB10 BSPB21 BSPB31 BSPB41 BSPB51 BSPB61 BSPB72 BSPB80 BSPC10 BSPC21 BSPC32 BSPC42 BSPC51 BSPC61 BSPC72 BSPC80 BSPD10 BSPD21 BSPD32 BSPD41 BSPD52 BSPD61 BSPD70 BSPD82 BSPE11 BSPE22 BSPE31 BSPE42 BSPE52 BSPE61 BSPE72 BSPE82 BSPF12 BSPF21 BSPF32 BSPF41 BSPF52 BSPF62 BSPF71 BSPF82 BSPG10 BSPG22 BSPG31 BSPG42 BSPG52 BSPG61 BSPG72 BSPG82 BSPH12 BSPH21 BSPH30 BSPH42 BSPH51 BSPH61 BSPH70 BSPH82 BSPA2B BSPA4B BSPB2B BSPB3B BSPB4B BSPB5B BSPB6B BSPB7W BSPC2B BSPC3W BSPC4W BSPC5B BSPC6B BSPC7W BSPD2B BSPD3W BSPD4B BSPD5W BSPD6B BSPD8W BSPE1B BSPE2W BSPE3B BSPE4W BSPE5W BSPE6B BSPE7W BSPE8W BSPF1W BSPF2B BSPF3W BSPF4B BSPF5W BSPF6W BSPF7B BSPF8W BSPG2W BSPG3B BSPG4W BSPG5W BSPG6B BSPG7

In [5]:
# Resumen: tabla BSP x posición (20x198) — solo BSPs de piezas (no vacías)
piece_mask = np.array([len(n) == 6 and n.startswith('BSP') and not n.endswith('0') for n in bsp_names])
piece_names = bsp_names[piece_mask]          # 128 BSPs
last15_piece_bsp = last15_bsp[:, piece_mask] # (15, 128)

print('Matriz BSPs de piezas (15 posiciones x 128 BSPs)')
print(f'Shape: {last15_piece_bsp.shape}')
print(f'Activos por posición: {last15_piece_bsp.sum(axis=1).tolist()}')
print(f'Posiciones activas por BSP (top 10):')
counts = last15_piece_bsp.sum(axis=0)
top10  = np.argsort(counts)[::-1][:10]
for idx in top10:
    print(f'  {piece_names[idx]}: {int(counts[idx])}/15 posiciones')

Matriz BSPs de piezas (15 posiciones x 128 BSPs)
Shape: (15, 256)
Activos por posición: [98, 100, 102, 104, 106, 108, 110, 112, 114, 116, 118, 120, 122, 124, 126]
Posiciones activas por BSP (top 10):
  BSPH8W: 15/15 posiciones
  BSPH6B: 15/15 posiciones
  BSPH5B: 15/15 posiciones
  BSPH4W: 15/15 posiciones
  BSPH1W: 15/15 posiciones
  BSPG5W: 15/15 posiciones
  BSPG7W: 15/15 posiciones
  BSPG8W: 15/15 posiciones
  BSPG6B: 15/15 posiciones
  BSPE7W: 15/15 posiciones


In [6]:
import pandas as pd

piece_mask = np.array([len(n) == 6 and n.startswith('BSP') and not n.endswith('0') for n in bsp_names])
piece_names = bsp_names[piece_mask]
last15_piece_bsp = last15_bsp[:, piece_mask]  # (15, 128)

df = pd.DataFrame(
    last15_piece_bsp,
    columns=piece_names,
    index=[f'Pos_{i}' for i in range(15)]
)

print(f'DataFrame de las 15 posiciones:')
print('=' * 60)
print(df)

DataFrame de las 15 posiciones:
        BSPA11  BSPA12  BSPA21  BSPA22  BSPA31  BSPA32  BSPA41  BSPA42  \
Pos_0        0       0       1       0       0       0       1       0   
Pos_1        0       0       0       1       0       0       0       1   
Pos_2        0       0       1       0       0       0       1       0   
Pos_3        0       0       0       1       0       0       0       1   
Pos_4        0       0       1       0       0       0       1       0   
Pos_5        0       0       0       1       0       0       0       1   
Pos_6        0       0       1       0       0       0       1       0   
Pos_7        0       0       0       1       0       0       0       1   
Pos_8        0       0       1       0       0       0       1       0   
Pos_9        0       0       0       1       0       0       0       1   
Pos_10       0       0       1       0       0       0       1       0   
Pos_11       0       0       0       1       0       0       0       1   
Pos_12

In [7]:
BSP_OBJETIVO = "BSPG4B"

print(f'Ground truth de {BSP_OBJETIVO} en los 15 tableros:')
print('=' * 40)
print(df[[BSP_OBJETIVO]])

Ground truth de BSPG4B en los 15 tableros:
        BSPG4B
Pos_0        0
Pos_1        0
Pos_2        0
Pos_3        0
Pos_4        0
Pos_5        0
Pos_6        0
Pos_7        0
Pos_8        1
Pos_9        1
Pos_10       1
Pos_11       1
Pos_12       1
Pos_13       1
Pos_14       1
